# Precision without Mean Field

This example is part of the *Universal Inference* section, which explores running inference in a broad class of models by approximating what has no closed form. Here every node has a closed form under mean field, and we drop the factorisation instead: the classic model of a Gaussian with unknown mean and precision,

$$\begin{aligned}
x &\sim \mathcal{N}(m_0, v_0), \qquad \tau \sim \Gamma(a_0, b_0),\\
y_i &\sim \mathcal{N}(x, \tau^{-1}), \quad i = 1, \dots, N,
\end{aligned}$$

with $\Gamma(a, b)$ the Gamma distribution of shape $a$ and rate $b$. Under the mean-field factorisation $q(x, \tau) = q(x)\,q(\tau)$, variational message passing has closed-form updates, and is known to be overconfident about $\tau$ (Bishop, 2006, Section 10.1.3). Belief propagation does not factorise, and its messages have no closed form. The message of an observation towards $\tau$ integrates the likelihood against the message of $x$, a Gaussian $\mathcal{N}(m_x, v_x)$:

$$\ell_{i \to \tau}(\tau) = \log \int \mathcal{N}(y_i \mid x, \tau^{-1})\, \mathcal{N}(x \mid m_x, v_x)\, dx = \log \mathcal{N}(y_i \mid m_x, v_x + \tau^{-1}),$$

which is not of the Gamma family: next to $\frac{1}{2}\log\tau$ it has the term $-\frac{1}{2}\log(1 + v_x\tau)$, the uncertainty of $x$, which variational message passing leaves out. `fallback = NaturalGradientPropagation()`, natural-gradient message passing (Lukashchuk et al., 2026), gives each such message a Gamma site, which moves at each iteration by a damped step towards its target, $\eta^\star = G(q)^{-1}\,\mathrm{Cov}_q[T(\tau), \ell(\tau)]$, with $T(\tau) = (\log\tau, \tau)$ the Gamma's sufficient statistics and $G$ its Fisher information under the marginal $q(\tau)$. At the fixed point, every site is its target. This model is the paper's example of vanishing edge uncertainty; we compare both schemes with the exact posterior on a grid.

In [ ]:
using RxInfer, BayesBase, Distributions, LinearAlgebra, StableRNGs
using Plots, PrettyTables, Printf

## Data

The priors are the paper's, $x \sim \mathcal{N}(0, 25)$ and $\tau \sim \Gamma(2, 1)$. We draw a true mean and precision from them, and ten observations.

In [ ]:
m0, v0, a0, b0 = 0.0, 25.0, 2.0, 1.0

function draw(rng, N)
    x = rand(rng, Normal(m0, sqrt(v0)))
    τ = rand(rng, Gamma(a0, 1 / b0))
    return (; x, τ, y = x .+ randn(rng, N) ./ sqrt(τ))
end

data = draw(StableRNG(7), 10)
y = data.y
@printf("true x = %.3f, true τ = %.3f, N = %d\n", data.x, data.τ, length(y))

## The exact posterior

On a grid of $(x, \log\tau)$, 400 points each way, the log-density of the joint gives the exact marginals of $x$ and $\tau$ and the evidence $p(y)$. The grid spans ten standard deviations on each side of a first estimate of the posterior, from the sample mean and variance.

In [ ]:
logjoint(x, τ, y) = logpdf(Normal(m0, sqrt(v0)), x) + logpdf(Gamma(a0, 1 / b0), τ) + sum(yi -> logpdf(Normal(x, 1 / sqrt(τ)), yi), y)

function exact_posterior(y; n = 400)
    N, ȳ = length(y), sum(y) / length(y)
    s² = sum(abs2, y .- ȳ) / N
    a, b = a0 + N / 2, b0 + N * s² / 2
    sd_u, τ̂ = 1 / sqrt(a), a / b
    half = 10 / sqrt(1 / v0 + N * τ̂ * exp(-3sd_u))
    xs = range(ȳ - half, ȳ + half, length = n)
    us = range(log(τ̂) - 10sd_u, log(τ̂) + 10sd_u, length = n)
    L = [logjoint(x, exp(u), y) + u for x in xs, u in us]
    P = exp.(L .- maximum(L))
    Z = sum(P)
    hx, hu = step(xs), step(us)
    px, pu = vec(sum(P; dims = 2)) ./ (Z * hx), vec(sum(P; dims = 1)) ./ (Z * hu)
    τs = exp.(us)
    Eτ = sum(pu .* τs) * hu
    return (;
        xs, us, px, pu, logevidence = maximum(L) + log(Z * hx * hu),
        x = (mean = sum(px .* xs) * hx, sd = sqrt(sum(px .* xs .^ 2) * hx - (sum(px .* xs) * hx)^2)),
        τ = (mean = Eτ, sd = sqrt(sum(pu .* τs .^ 2) * hu - Eτ^2)),
    )
end

# The inclusive divergences KL(p ‖ q) of the marginals, on the grid: τ's in log τ, with the Jacobian
function kl_x(exact, q)
    h = step(exact.xs)
    return sum(p * (log(p) - logpdf(q, x)) for (p, x) in zip(exact.px, exact.xs) if p > 0) * h
end
function kl_τ(exact, q)
    h = step(exact.us)
    return sum(p * (log(p) - logpdf(q, exp(u)) - u) for (p, u) in zip(exact.pu, exact.us) if p > 0) * h
end

exact = exact_posterior(y)
@printf("exact: E[x] = %.4f, sd(x) = %.4f, E[τ] = %.4f, sd(τ) = %.4f, log p(y) = %.4f\n", exact.x.mean, exact.x.sd, exact.τ.mean, exact.τ.sd, exact.logevidence)

## Mean field: variational message passing

With RxInfer's own `NormalMeanPrecision` node and the constraint $q(x, \tau) = q(x)\,q(\tau)$, every message has a closed form, and the iterations need a start for both marginals.

In [ ]:
@model function normal_model(y, likelihood)
    x ~ NormalMeanVariance(m0, v0)
    τ ~ Gamma(shape = a0, rate = b0)
    for i in eachindex(y)
        y[i] ~ likelihood(x, τ)
    end
end

vmp = infer(
    model = normal_model(likelihood = NormalMeanPrecision), data = (y = y,),
    constraints = MeanField(), iterations = 100, free_energy = true,
    initialization = @initialization(begin
        q(x) = NormalMeanVariance(m0, v0)
        q(τ) = GammaShapeRate(a0, b0)
    end),
)
q_vmp = (x = last(vmp.posteriors[:x]), τ = last(vmp.posteriors[:τ]))

## Natural-gradient propagation, with RxInfer's Normal node

Without a factorisation, the same model goes to `fallback = NaturalGradientPropagation()`. `NormalMeanPrecision` has rules for mean field only, so the fallback computes both of its messages. $x$ and $\tau$ are connected through every observation, a loop, which belief propagation starts from initial messages: we give each variable its prior. Each of the twenty sites takes one step per iteration, with the default step size `alpha = 0.5` and momentum `beta = 0.2`; 60 iterations are enough for all of them to settle.

In [ ]:
initial_messages = @initialization begin
    μ(x) = NormalMeanVariance(m0, v0)
    μ(τ) = GammaShapeRate(a0, b0)
end

fit(likelihood, y) = infer(
    model = normal_model(likelihood = likelihood), data = (y = y,),
    initialization = initial_messages, iterations = 60,
    fallback = NaturalGradientPropagation(), free_energy = true, report = true,
)

ngmp_native = fit(NormalMeanPrecision, y)
q_native = (x = last(ngmp_native.posteriors[:x]), τ = last(ngmp_native.posteriors[:τ]))

The report shows each variable's sites and what formed them. Towards $\tau$, the fallback sends belief propagation's message $\ell_{i \to \tau}$, against the Gaussian message of $x$: $\tau$'s ten sites are Gamma distributions. Towards $x$, belief propagation's message would integrate the likelihood against the message of $\tau$, a Gamma distribution, which the fallback does not do: it averages the log-likelihood over that message instead, as variational message passing does with the marginal. The report warns about it.

In [ ]:
@assert [d.kind for d in RxInfer.diagnostics(ngmp_native.report)] == [:expectation_over_messages] # hide
@assert ngmp_native.report.approximations.variables[(:τ, nothing)].sites.outcome.method === :natural_gradient # hide
@assert q_native.τ isa GammaShapeRate # hide
ngmp_native.report.approximations

## Belief propagation's message towards the mean

Integrated against a Gamma message $\Gamma(a, b)$ of $\tau$, the likelihood of $y_i$ is a Student-t density in $x$, with $2a$ degrees of freedom, centred at $y_i$ with scale $\sqrt{b / a}$. We write the likelihood as a node of its own, by its log-density, and give it this one message as a rule; the message towards $\tau$ remains the fallback's. The Student-t is not a Gaussian, so it gets a Gaussian site of its own, which steps towards its target like the others.

In [ ]:
struct PrecisionObservation{X, T} <: ContinuousUnivariateDistribution
    x::X
    τ::T
end

BayesBase.logpdf(d::PrecisionObservation, y) = (log(d.τ) - log(2π) - d.τ * (y - d.x)^2) / 2
BayesBase.insupport(::PrecisionObservation, y) = true

@define_factor_node(node = PrecisionObservation, type = Stochastic, interfaces = [:out, :x, :τ])

# Belief propagation's message towards x: ∫ N(y | x, 1/τ) Γ(τ | a, b) dτ, a Student-t in x
@define_message_update_rule(
    node = PrecisionObservation, target = :x,
    args = (q[:out]::PointMass, m[:τ]::GammaDistributionsFamily),
    body = (args) -> begin
        a, b = shape(args.m[:τ]), rate(args.m[:τ])
        return LocationScale(mean(args.q[:out]), sqrt(b / a), TDist(2a))
    end,
)

ngmp = fit(PrecisionObservation, y)
q_ngmp = (x = last(ngmp.posteriors[:x]), τ = last(ngmp.posteriors[:τ]))

The report now finds every message of the observations towards $x$ in the rule, and nothing to warn about:

In [ ]:
@assert isempty(RxInfer.diagnostics(ngmp.report)) # hide
@assert RxInfer.node_types(ngmp.report.approximations)[(PrecisionObservation, :message, :x)].rules == 10 * 60 # hide
ngmp.report.approximations

## Results

In [ ]:
row(name, q, F) = [name mean(q.x) std(q.x) mean(q.τ) std(q.τ) kl_x(exact, q.x) kl_τ(exact, q.τ) F]
rows = [
    row("mean field (VMP)", q_vmp, last(vmp.free_energy));
    row("NGMP, Normal node", q_native, last(ngmp_native.free_energy));
    row("NGMP, BP towards x", q_ngmp, last(ngmp.free_energy));
    "exact" exact.x.mean exact.x.sd exact.τ.mean exact.τ.sd 0.0 0.0 -exact.logevidence
]
@assert kl_x(exact, q_ngmp.x) < kl_x(exact, q_vmp.x) / 2 && kl_x(exact, q_native.x) > kl_x(exact, q_vmp.x) # hide
@assert 0.97 * exact.x.sd < std(q_ngmp.x) < exact.x.sd && std(q_vmp.x) < 0.93 * exact.x.sd # hide
@assert std(q_vmp.τ) < exact.τ.sd < std(q_ngmp.τ) && kl_τ(exact, q_ngmp.τ) < kl_τ(exact, q_vmp.τ) # hide
@assert last(vmp.free_energy) > -exact.logevidence > last(ngmp.free_energy) # hide
pretty_table(rows; column_labels = ["", "E[x]", "sd(x)", "E[τ]", "sd(τ)", "KL(p ‖ q), x", "KL(p ‖ q), τ", "free energy"], formatters = [fmt__printf("%.4f", 2:5), fmt__printf("%.2e", 6:7), fmt__printf("%.4f", 8:8)], fit_table_in_display_horizontally = false)

- **The mean.** With belief propagation's message towards $x$, the spread of its posterior is within 3% of the exact one. Mean field's is 8% narrower: it is $\mathcal{N}$ with the precision $1/v_0 + N\,\mathbb{E}[\tau]$, as if $\tau$ were known. With RxInfer's Normal node, the messages towards $x$ average over the message of $\tau$ on each edge, which leaves out that observation, and the posterior of $x$ is further from the exact one than mean field's.
- **The precision.** Mean field's $q(\tau)$ is too narrow, and natural-gradient propagation's a little too wide; on this data set, the latter is the closer one.
- **The free energy.** Mean field's is a bound, above $-\log p(y)$. Natural-gradient propagation's is Bethe's, with the average energies of the nodes without rules. The model has a loop through the observations, so it is a diagnostic of the fixed point, not a bound, and here it is below $-\log p(y)$.

In [ ]:
τs = exp.(exact.us)
plot(τs, exact.pu ./ τs; color = :black, linewidth = 2, label = "exact", xlabel = "τ", ylabel = "density", xlims = (0, quantile(q_ngmp.τ, 0.9999)))
plot!(τs, pdf.(q_vmp.τ, τs); color = :orange, linewidth = 2, label = "mean field (VMP)")
plot!(τs, pdf.(q_ngmp.τ, τs); color = :blue, linewidth = 2, label = "NGMP, BP towards x")
vline!([data.τ]; color = :gray, linestyle = :dot, label = "true τ")

## As the data grow

Lukashchuk et al. measure how the inclusive divergence $\mathrm{KL}(p \,\|\, q)$ of each marginal falls with the number of observations, for instances whose true mean and precision are drawn from the priors. We repeat it for $N$ from 4 to 64, with ten instances each.

In [ ]:
Ns = [4, 8, 16, 32, 64]
instances = 10

function divergences(N, k)
    d = draw(StableRNG(1000N + k), N)
    reference = exact_posterior(d.y)
    v = infer(
        model = normal_model(likelihood = NormalMeanPrecision), data = (y = d.y,), constraints = MeanField(), iterations = 100,
        initialization = @initialization(begin
            q(x) = NormalMeanVariance(m0, v0)
            q(τ) = GammaShapeRate(a0, b0)
        end),
    )
    n = fit(PrecisionObservation, d.y)
    qv, qn = (last(v.posteriors[:x]), last(v.posteriors[:τ])), (last(n.posteriors[:x]), last(n.posteriors[:τ]))
    settled = isempty(RxInfer.diagnostics(n.report))
    return (; vmp = (kl_x(reference, qv[1]), kl_τ(reference, qv[2])), ngmp = (kl_x(reference, qn[1]), kl_τ(reference, qn[2])), shapes = (shape(qv[2]), shape(qn[2]), reference.τ.mean^2 / reference.τ.sd^2), settled)
end

results = [divergences(N, k) for N in Ns, k in 1:instances]
@assert all(r -> r.settled, results) # hide
average(f) = [sum(f, results[j, :]) / instances for j in eachindex(Ns)]
kl = (vmp_x = average(r -> r.vmp[1]), vmp_τ = average(r -> r.vmp[2]), ngmp_x = average(r -> r.ngmp[1]), ngmp_τ = average(r -> r.ngmp[2]))

# the slope of log KL against log N, by least squares
slope(values) = (A = [ones(length(Ns)) log.(Ns)]; (A \ log.(values))[2])
@printf("slopes of log KL in log N: τ: VMP %.2f, NGMP %.2f;  x: VMP %.2f, NGMP %.2f\n", slope(kl.vmp_τ), slope(kl.ngmp_τ), slope(kl.vmp_x), slope(kl.ngmp_x))
@printf("KL of x, VMP / NGMP at N = %s: %s\n", join(Ns, ", "), join([@sprintf("%.1f", r) for r in kl.vmp_x ./ kl.ngmp_x], ", "))
@printf("KL of τ, VMP / NGMP at N = %s: %s\n", join(Ns, ", "), join([@sprintf("%.1f", r) for r in kl.vmp_τ ./ kl.ngmp_τ], ", "))

In [ ]:
p_x = plot(Ns, kl.vmp_x; marker = :circle, color = :orange, label = "mean field (VMP)", xscale = :log2, yscale = :log10, xlabel = "N", ylabel = "KL(p ‖ q)", title = "x")
plot!(p_x, Ns, kl.ngmp_x; marker = :circle, color = :blue, label = "NGMP, BP towards x")
p_τ = plot(Ns, kl.vmp_τ; marker = :circle, color = :orange, label = "mean field (VMP)", xscale = :log2, yscale = :log10, xlabel = "N", ylabel = "KL(p ‖ q)", title = "τ")
plot!(p_τ, Ns, kl.ngmp_τ; marker = :circle, color = :blue, label = "NGMP, BP towards x")
plot!(p_τ, Ns, kl.vmp_τ[end] .* (Ns ./ Ns[end]) .^ -2; color = :gray, linestyle = :dot, label = "∝ N⁻²")
plot!(p_τ, Ns, kl.vmp_τ[end] .* (Ns ./ Ns[end]) .^ -3; color = :gray, linestyle = :dash, label = "∝ N⁻³")
plot(p_x, p_τ; size = (1000, 400), bottom_margin = 5Plots.mm, left_margin = 5Plots.mm)

The grey lines fall as $N^{-2}$ and $N^{-3}$ through mean field's divergence at $N = 64$. For the mean, natural-gradient propagation is closer to the exact posterior at every $N$, by a factor of about two, and both divergences fall at the same rate. For the precision, both fall at about the same rate too, close to $N^{-2}$ (the fitted slopes between $N = 4$ and $64$ are above $-2$, the small $N$ not yet in the limit), and natural-gradient propagation is not ahead: on average its divergence is the larger one, twice mean field's at $N = 4$, within 20% of it at $N = 64$. The shapes of the Gamma distributions say why. The exact posterior of $\tau$ is close to a Gamma of shape $a_0 + (N - 1)/2$: integrating out $x$ costs one half. Mean field's shape is $a_0 + N/2$ exactly, half a unit too many. Natural-gradient propagation's is close to $a_0 + N/2 - 1$, half a unit too few: each site $\ell_{i \to \tau}$ sees $x$'s uncertainty, through $v_x$, but the $N$ sites see it separately, and together they take away twice as much as the joint integral does. At $N = 64$, averaged over the instances:

In [ ]:
j = findfirst(==(64), Ns)
shapes = [sum(r -> r.shapes[k], results[j, :]) / instances for k in 1:3]
@printf("shape of q(τ) at N = 64: VMP %.3f, NGMP %.3f, exact (moment-matched) %.3f;  a₀ + N/2 = %.1f\n", shapes[1], shapes[2], shapes[3], a0 + 32)
@assert all(kl.ngmp_x .< kl.vmp_x ./ 1.5) # hide
@assert all(-2.1 < s < -1.5 for s in (slope(kl.vmp_τ), slope(kl.ngmp_τ), slope(kl.vmp_x), slope(kl.ngmp_x))) # hide
@assert all(kl.ngmp_τ .> kl.vmp_τ) && kl.ngmp_τ[1] > 1.8 * kl.vmp_τ[1] && kl.ngmp_τ[end] < 1.2 * kl.vmp_τ[end] # hide
@assert all(0.5 .< kl.ngmp_τ ./ kl.vmp_τ .< 2.0) # hide
@assert abs(shapes[1] - (a0 + 32)) < 1.0e-6 && abs(shapes[2] - (a0 + 31)) < 0.2 && abs(shapes[3] - (a0 + 31.5)) < 0.2 # hide

Lukashchuk et al. report that the divergence of the precision falls as $N^{-3}$ under natural-gradient propagation, against $N^{-2}$ under mean field; with the messages above, belief propagation's in both directions, we find the same rate for both, and the shapes above suggest $N^{-2}$ for both. Where natural-gradient propagation pays off in this model is the mean, whose posterior keeps the uncertainty of $\tau$.

## Summary

- Without mean field, `fallback = NaturalGradientPropagation()` gives the precision Gamma sites from belief propagation's messages, which carry the uncertainty of the mean. Towards the mean, the fallback averages over the message of the precision instead of integrating against it, and says so in the report; one rule, the Student-t, makes that message belief propagation's too.
- With both messages belief propagation's, the posterior of the mean is about twice as close to the exact one as mean field's in divergence, at every $N$ we tried. The posterior of the precision is a little too wide where mean field's is too narrow, and on average no closer: both divergences fall as about $N^{-2}$.

## References

- M. Lukashchuk, K. Yemets, A. Ledbetter and İ. Şenöz (2026). Information geometry of message passing. arXiv:2608.15922.
- C. M. Bishop (2006). *Pattern Recognition and Machine Learning*, Section 10.1.3. Springer.